# AI-Enhanced Predictive Risk Management in Agile IT Projects
### MSc IT with Project Management | University of the West of Scotland
**Student:** Manfred Oppong | **Banner ID:** B01814357 | **Supervisor:** Durfashan Tariq | **Year:** 2026

---
This notebook provides a complete end-to-end analysis pipeline for the dissertation artefact:
*"The Role of Artificial Intelligence in Enhancing Predictive Risk Management in Agile IT Projects"*.
It covers data loading, exploratory data analysis, feature engineering, model training, evaluation, and a live prediction demo.


## Cell 1 — Environment Setup & Library Installation

Install and import all required libraries. Non-standard packages are installed via `!pip install`.
Global settings for matplotlib, warnings, and random seeds are configured here.


In [ ]:
# ── Install non-standard libraries ──────────────────────────────────────────
import subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

required_pkgs = [
    "ttkbootstrap",
    "scikit-learn",
    "pandas",
    "numpy",
    "matplotlib",
    "seaborn",
    "joblib",
    "openpyxl",
    "imbalanced-learn",
    "xgboost",
    "scipy",
]
for pkg in required_pkgs:
    try:
        __import__(pkg.replace("-", "_").split(">=")[0].split("==")[0])
    except ImportError:
        print(f"Installing {pkg}...")
        pip_install(pkg)

print("✅ All packages available.")


In [ ]:
# ── Standard library imports ────────────────────────────────────────────────
import os
import zipfile
import pathlib
import warnings
import itertools
import glob
import json
import re
import datetime

# ── Numerical / data ────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ───────────────────────────────────────────────────────────
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.cm as cm
import seaborn as sns

# ── Scikit-learn ─────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder, label_binarize
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    log_loss, roc_curve, auc
)
from sklearn.impute import SimpleImputer

# ── Misc ─────────────────────────────────────────────────────────────────────
import joblib
from scipy import stats
from scipy.stats import pearsonr

# ── Settings ─────────────────────────────────────────────────────────────────
warnings.filterwarnings("ignore")
np.random.seed(42)
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("seaborn-whitegrid")

matplotlib.rcParams["figure.dpi"] = 150
matplotlib.rcParams["figure.figsize"] = (10, 5)
matplotlib.rcParams["axes.titlesize"] = 12
matplotlib.rcParams["axes.labelsize"] = 10

# ── Output directory ──────────────────────────────────────────────────────────
OUTPUT_DIR = pathlib.Path("/content/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONTENT_DIR = pathlib.Path("/content")
print(f"✅ Environment ready. Output dir: {OUTPUT_DIR}")
print(f"   NumPy  {np.__version__} | Pandas {pd.__version__} | Matplotlib {matplotlib.__version__}")


## Cell 2 — Data Loading

Scan `/content` for `.zip` archives and auto-extract them, then load all **19 datasets** using their
exact filenames and variable names as specified in the dataset inventory.
A summary table is printed showing variable name, source file, shape, and total null count.


In [ ]:
# ── Auto-extract any ZIP files in /content ──────────────────────────────────
for zf in glob.glob(str(CONTENT_DIR / "*.zip")):
    dest = CONTENT_DIR
    print(f"Extracting {zf} → {dest}")
    with zipfile.ZipFile(zf, "r") as z:
        z.extractall(dest)
    print(f"  Done.")

# ── Helper ────────────────────────────────────────────────────────────────────
def load_csv(name, **kwargs):
    """Load CSV from /content or any immediate sub-directory."""
    candidates = list(CONTENT_DIR.rglob(name))
    if not candidates:
        print(f"  ⚠️  File not found: {name}")
        return None
    path = candidates[0]
    try:
        df = pd.read_csv(path, **kwargs)
        return df
    except Exception as e:
        print(f"  ⚠️  Could not load {name}: {e}")
        return None

def load_excel(name, **kwargs):
    """Load Excel file from /content or any immediate sub-directory."""
    candidates = list(CONTENT_DIR.rglob(name))
    if not candidates:
        print(f"  ⚠️  File not found: {name}")
        return None
    path = candidates[0]
    try:
        df = pd.read_excel(path, **kwargs)
        return df
    except Exception as e:
        print(f"  ⚠️  Could not load {name}: {e}")
        return None

print("Loading datasets...")

# ── GROUP A — NASA PROMISE Defect Datasets ────────────────────────────────────
cm1  = load_csv("cm1.csv")
pc1  = load_csv("pc1.csv")
jm1  = load_csv("jm1.csv")
kc1  = load_csv("kc1.csv")
kc2  = load_csv("kc2.csv")

# ── GROUP B — Project Risk Dataset ───────────────────────────────────────────
project_risk_raw_dataset = load_csv("project_risk_raw_dataset.csv")

# ── GROUP C — Mesos ───────────────────────────────────────────────────────────
Mesos_Stories_176      = load_csv("Mesos Stories 176.csv")
MESO_Issue_Summary_370 = load_csv("MESO Issue Summary 370.csv")
MESO_Sprint_96         = load_csv("MESO Sprint 96.csv")

# ── GROUP C — Spring XD ───────────────────────────────────────────────────────
Spring_XD_Issues_1992          = load_csv("Spring XD Issues 1992.csv")
Spring_XD_Issues_Summary_2861  = load_csv("Spring XD Issues Summary 2861.csv")
Spring_XD_Sprints_67           = load_csv("Spring XD Sprints 67.csv")

# ── GROUP C — Aurora ─────────────────────────────────────────────────────────
Aurora_Issues_554          = load_csv("Aurora Issues 554.csv")
Aurora_Issues_summery_568  = load_csv("Aurora Issues summery 568.csv")
Aurora_Sprints_41          = load_csv("Aurora Sprints 41.csv")

# ── GROUP C — Usergrid ────────────────────────────────────────────────────────
Usergrid_Issues_824          = load_csv("Usergrid Issues 824.csv")
Usergrid_Issues_Summary_929  = load_csv("Usergrid Issues Summary 929.csv")
Usergrid_Sprints_36          = load_csv("Usergrid Sprints 36.csv")

# ── GROUP D — Agile Projects ──────────────────────────────────────────────────
Agile_Projects_Dataset = load_excel("Agile_Projects_Dataset.xlsx")

# ── Summary Table ─────────────────────────────────────────────────────────────
dataset_inventory = [
    ("cm1",                          "cm1.csv",                         cm1),
    ("pc1",                          "pc1.csv",                         pc1),
    ("jm1",                          "jm1.csv",                         jm1),
    ("kc1",                          "kc1.csv",                         kc1),
    ("kc2",                          "kc2.csv",                         kc2),
    ("project_risk_raw_dataset",     "project_risk_raw_dataset.csv",    project_risk_raw_dataset),
    ("Mesos_Stories_176",            "Mesos Stories 176.csv",           Mesos_Stories_176),
    ("MESO_Issue_Summary_370",       "MESO Issue Summary 370.csv",      MESO_Issue_Summary_370),
    ("MESO_Sprint_96",               "MESO Sprint 96.csv",              MESO_Sprint_96),
    ("Spring_XD_Issues_1992",        "Spring XD Issues 1992.csv",       Spring_XD_Issues_1992),
    ("Spring_XD_Issues_Summary_2861","Spring XD Issues Summary 2861.csv",Spring_XD_Issues_Summary_2861),
    ("Spring_XD_Sprints_67",         "Spring XD Sprints 67.csv",        Spring_XD_Sprints_67),
    ("Aurora_Issues_554",            "Aurora Issues 554.csv",           Aurora_Issues_554),
    ("Aurora_Issues_summery_568",    "Aurora Issues summery 568.csv",   Aurora_Issues_summery_568),
    ("Aurora_Sprints_41",            "Aurora Sprints 41.csv",           Aurora_Sprints_41),
    ("Usergrid_Issues_824",          "Usergrid Issues 824.csv",         Usergrid_Issues_824),
    ("Usergrid_Issues_Summary_929",  "Usergrid Issues Summary 929.csv", Usergrid_Issues_Summary_929),
    ("Usergrid_Sprints_36",          "Usergrid Sprints 36.csv",         Usergrid_Sprints_36),
    ("Agile_Projects_Dataset",       "Agile_Projects_Dataset.xlsx",     Agile_Projects_Dataset),
]

summary_rows = []
for var_name, file_name, df in dataset_inventory:
    if df is not None:
        rows, cols = df.shape
        nulls = int(df.isnull().sum().sum())
        summary_rows.append({"Variable": var_name, "File": file_name,
                              "Rows": rows, "Columns": cols, "Total Nulls": nulls})
    else:
        summary_rows.append({"Variable": var_name, "File": file_name,
                              "Rows": "NOT FOUND", "Columns": "-", "Total Nulls": "-"})

summary_df = pd.DataFrame(summary_rows)
print("\n📦 Dataset Loading Summary:")
print(summary_df.to_string(index=False))
loaded = sum(1 for _, _, df in dataset_inventory if df is not None)
print(f"\n✅ {loaded}/19 datasets loaded successfully.")


## Cell 3 — Exploratory Data Analysis: NASA PROMISE Defect Datasets

Analyse the five NASA software defect datasets (cm1, pc1, jm1, kc1, kc2).
- Descriptive statistics via `.describe()`
- Class distribution (defective vs non-defective) across all five datasets
- Pearson correlation heatmap for cm1
- LOC (lines of code) distribution via violin plots
- Combined defect rate across all five datasets


In [ ]:
# ── Descriptive Statistics ────────────────────────────────────────────────────
nasa_datasets = {
    "cm1": cm1, "pc1": pc1, "jm1": jm1, "kc1": kc1, "kc2": kc2
}

for name, df in nasa_datasets.items():
    if df is None:
        print(f"{name}: NOT LOADED\n")
        continue
    print(f"\n{'='*60}")
    print(f"  {name.upper()} — shape: {df.shape}")
    print(f"{'='*60}")
    print(df.describe().T.round(3))


In [ ]:
# ── Normalise target columns ─────────────────────────────────────────────────
def get_target(df, name):
    """Return binary 0/1 target series from each NASA dataset."""
    if df is None:
        return None
    if name == "kc2":
        col = "problems"
        return (df[col].astype(str).str.lower().map({"yes": 1, "no": 0, "true": 1, "false": 0})
                .fillna(0).astype(int))
    else:
        col = "defects"
        if col not in df.columns:
            # try boolean / object
            possible = [c for c in df.columns if "defect" in c.lower() or "problem" in c.lower()]
            col = possible[0] if possible else None
        if col is None:
            return None
        s = df[col]
        if s.dtype == bool or set(s.dropna().unique()).issubset({True, False}):
            return s.astype(int)
        return (s.astype(str).str.lower().map({"true": 1, "false": 0, "yes": 1, "no": 0})
                .fillna(0).astype(int))

targets = {n: get_target(d, n) for n, d in nasa_datasets.items()}


In [ ]:
# ── Class Distribution — 1×5 horizontal bar chart ────────────────────────────
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
fig.suptitle("NASA Defect Dataset — Class Distribution (Defect=1 vs Non-Defect=0)",
             fontsize=13, fontweight="bold", y=1.02)

colors = ["#2ECC71", "#E74C3C"]
for ax, (name, df) in zip(axes, nasa_datasets.items()):
    if df is None or targets[name] is None:
        ax.set_title(f"{name}\n(Not Loaded)")
        continue
    t = targets[name]
    counts = t.value_counts().reindex([0, 1], fill_value=0)
    labels = ["No Defect", "Defect"]
    ax.barh(labels, counts.values, color=colors)
    ax.set_title(f"{name.upper()}\n(n={len(t):,})")
    ax.set_xlabel("Count")
    for i, v in enumerate(counts.values):
        ax.text(v + 1, i, str(v), va="center", fontsize=9)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "nasa_class_distribution.png", bbox_inches="tight")
plt.show()


In [ ]:
# ── Pearson Correlation Heatmap — cm1 ────────────────────────────────────────
if cm1 is not None and targets["cm1"] is not None:
    cm1_corr = cm1.copy()
    cm1_corr["defect_binary"] = targets["cm1"]
    numeric_cols = cm1_corr.select_dtypes(include=[np.number]).columns.tolist()
    corr_matrix = cm1_corr[numeric_cols].corr(method="pearson")

    fig, ax = plt.subplots(figsize=(14, 11))
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap="RdYlGn",
                linewidths=0.5, ax=ax, annot_kws={"size": 7},
                vmin=-1, vmax=1, center=0)
    ax.set_title("cm1 — Pearson Correlation Heatmap (all numeric features + target)",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "cm1_correlation_heatmap.png", bbox_inches="tight")
    plt.show()


In [ ]:
# ── LOC (Lines of Code) Distribution — Violin Plots ─────────────────────────
loc_data = []
loc_labels = []
for name, df in nasa_datasets.items():
    if df is None:
        continue
    loc_col = "loc" if "loc" in df.columns else None
    if loc_col is None:
        possible = [c for c in df.columns if c.lower() == "loc"]
        loc_col = possible[0] if possible else None
    if loc_col:
        vals = df[loc_col].dropna().clip(upper=df[loc_col].quantile(0.99)).values
        loc_data.append(vals)
        loc_labels.append(name.upper())

if loc_data:
    fig, ax = plt.subplots(figsize=(12, 5))
    parts = ax.violinplot(loc_data, showmedians=True, showextrema=True)
    ax.set_xticks(range(1, len(loc_labels) + 1))
    ax.set_xticklabels(loc_labels, fontsize=10)
    ax.set_title("Lines of Code (LOC) Distribution Across NASA Datasets (clipped at 99th pctile)",
                 fontsize=12, fontweight="bold")
    ax.set_ylabel("LOC")
    ax.set_xlabel("Dataset")
    colors_v = ["#3498DB", "#2ECC71", "#E74C3C", "#F0A500", "#9B59B6"]
    for pc, color in zip(parts["bodies"], colors_v):
        pc.set_facecolor(color)
        pc.set_alpha(0.7)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "nasa_loc_violin.png", bbox_inches="tight")
    plt.show()


In [ ]:
# ── Combined Defect Rate across all 5 datasets ───────────────────────────────
print("\n📊 Defect Rate Summary across NASA Datasets")
print("-" * 45)
total_records = 0
total_defects = 0
for name, df in nasa_datasets.items():
    if df is None or targets[name] is None:
        print(f"  {name:6s}: NOT LOADED")
        continue
    t = targets[name]
    n = len(t)
    d = int(t.sum())
    rate = d / n * 100
    total_records += n
    total_defects += d
    print(f"  {name:6s}: {d:5d}/{n:6d}  ({rate:.1f}% defective)")

print("-" * 45)
if total_records > 0:
    print(f"  TOTAL : {total_defects:5d}/{total_records:6d}  ({total_defects/total_records*100:.1f}% defective)")


## Cell 4 — Exploratory Data Analysis: Agile Scrum Sprint Velocity Datasets

Analyse velocity trends, backlog growth, developer allocation, and story point distributions
across the four open-source Agile projects: **Mesos**, **Spring XD**, **Aurora**, and **Usergrid**.


In [ ]:
# ── Helper: normalise column name for total issues ───────────────────────────
def get_total_col(df):
    """Return the column name representing total issues in a sprint dataframe."""
    for c in ["totalNumberOfIssues", "total"]:
        if c in df.columns:
            return c
    return None

# ── Project configuration ─────────────────────────────────────────────────────
sprint_projects = {
    "Mesos":     (MESO_Sprint_96,         MESO_Issue_Summary_370),
    "Spring XD": (Spring_XD_Sprints_67,   Spring_XD_Issues_Summary_2861),
    "Aurora":    (Aurora_Sprints_41,       Aurora_Issues_summery_568),
    "Usergrid":  (Usergrid_Sprints_36,     Usergrid_Issues_Summary_929),
}

project_colors = {
    "Mesos": "#3498DB", "Spring XD": "#2ECC71",
    "Aurora": "#E74C3C", "Usergrid": "#9B59B6"
}


In [ ]:
# ── Velocity Trends — Line Charts ────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("Sprint Velocity Trends — Completed Issues per Sprint",
             fontsize=14, fontweight="bold")

for ax, (proj_name, (sprints_df, issues_df)) in zip(axes.flat, sprint_projects.items()):
    if sprints_df is None:
        ax.set_title(f"{proj_name} — NOT LOADED")
        continue
    df = sprints_df.copy()
    # Use completedIssuesEstimateSum if available, else completedIssuesCount
    if "completedIssuesEstimateSum" in df.columns:
        velocity_col = "completedIssuesEstimateSum"
    elif "completedIssuesCount" in df.columns:
        velocity_col = "completedIssuesCount"
    else:
        ax.set_title(f"{proj_name} — No velocity col")
        continue
    df = df.reset_index(drop=True)
    df["sprint_num"] = range(1, len(df) + 1)
    df[velocity_col] = pd.to_numeric(df[velocity_col], errors="coerce").fillna(0)
    color = project_colors[proj_name]
    ax.plot(df["sprint_num"], df[velocity_col], marker="o", color=color,
            linewidth=2, markersize=5, label="Velocity")
    # Rolling mean
    rolling = df[velocity_col].rolling(window=3, min_periods=1).mean()
    ax.plot(df["sprint_num"], rolling, linestyle="--", color="grey",
            linewidth=1.5, label="3-Sprint Rolling Mean")
    ax.set_title(f"{proj_name} — Sprint Velocity", fontsize=11, fontweight="bold")
    ax.set_xlabel("Sprint Number")
    ax.set_ylabel(velocity_col)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "sprint_velocity_trends.png", bbox_inches="tight")
plt.show()


In [ ]:
# ── Backlog Growth — Bar Charts ───────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("Sprint Backlog Growth — Issues Added During Sprint",
             fontsize=14, fontweight="bold")

for ax, (proj_name, (sprints_df, issues_df)) in zip(axes.flat, sprint_projects.items()):
    if sprints_df is None:
        ax.set_title(f"{proj_name} — NOT LOADED")
        continue
    df = sprints_df.copy()
    if "issueKeysAddedDuringSprint" not in df.columns:
        ax.set_title(f"{proj_name} — No backlog growth col")
        continue
    df = df.reset_index(drop=True)
    df["sprint_num"] = range(1, len(df) + 1)
    # Count pipe-separated keys if string
    def count_keys(val):
        if pd.isna(val) or val == "" or val == "[]":
            return 0
        if isinstance(val, str):
            # Handle JSON array format
            try:
                parsed = json.loads(val)
                return len(parsed)
            except Exception:
                return len([x for x in val.split(",") if x.strip()])
        try:
            return int(val)
        except Exception:
            return 0
    df["backlog_growth"] = df["issueKeysAddedDuringSprint"].apply(count_keys)
    color = project_colors[proj_name]
    ax.bar(df["sprint_num"], df["backlog_growth"], color=color, alpha=0.8, edgecolor="white")
    ax.axhline(df["backlog_growth"].mean(), color="red", linestyle="--",
               linewidth=1.5, label=f"Mean: {df['backlog_growth'].mean():.1f}")
    ax.set_title(f"{proj_name} — Backlog Growth per Sprint", fontsize=11, fontweight="bold")
    ax.set_xlabel("Sprint Number")
    ax.set_ylabel("Issues Added Mid-Sprint")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "sprint_backlog_growth.png", bbox_inches="tight")
plt.show()


In [ ]:
# ── NoOfDevelopers & SprintLength — Box Plots ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

dev_data, dev_labels = [], []
sprint_len_data, sprint_len_labels = [], []

for proj_name, (sprints_df, issues_df) in sprint_projects.items():
    if sprints_df is None:
        continue
    df = sprints_df.copy()
    if "NoOfDevelopers" in df.columns:
        vals = pd.to_numeric(df["NoOfDevelopers"], errors="coerce").dropna()
        if len(vals) > 0:
            dev_data.append(vals.values)
            dev_labels.append(proj_name)
    if "SprintLength" in df.columns:
        vals = pd.to_numeric(df["SprintLength"], errors="coerce").dropna()
        if len(vals) > 0:
            sprint_len_data.append(vals.values)
            sprint_len_labels.append(proj_name)

# Box plot: NoOfDevelopers
if dev_data:
    bp1 = axes[0].boxplot(dev_data, labels=dev_labels, patch_artist=True,
                          notch=False, vert=True)
    colors_list = [project_colors[l] for l in dev_labels]
    for patch, color in zip(bp1["boxes"], colors_list):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    axes[0].set_title("Number of Developers per Sprint", fontweight="bold")
    axes[0].set_ylabel("Developers")
    axes[0].grid(True, alpha=0.3)

# Box plot: SprintLength
if sprint_len_data:
    bp2 = axes[1].boxplot(sprint_len_data, labels=sprint_len_labels, patch_artist=True,
                          notch=False, vert=True)
    colors_list2 = [project_colors[l] for l in sprint_len_labels]
    for patch, color in zip(bp2["boxes"], colors_list2):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    axes[1].set_title("Sprint Length (Days)", fontweight="bold")
    axes[1].set_ylabel("Days")
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "sprint_dev_length_boxplots.png", bbox_inches="tight")
plt.show()


In [ ]:
# ── Average Completion Rate per Project ──────────────────────────────────────
print("\n📊 Average Completion Rate per Project")
print("-" * 50)
for proj_name, (sprints_df, issues_df) in sprint_projects.items():
    if sprints_df is None:
        print(f"  {proj_name}: NOT LOADED")
        continue
    df = sprints_df.copy()
    total_col = get_total_col(df)
    if total_col and "completedIssuesCount" in df.columns:
        df[total_col]              = pd.to_numeric(df[total_col], errors="coerce")
        df["completedIssuesCount"] = pd.to_numeric(df["completedIssuesCount"], errors="coerce")
        valid = df.dropna(subset=[total_col, "completedIssuesCount"])
        valid = valid[valid[total_col] > 0]
        if len(valid) > 0:
            cr = (valid["completedIssuesCount"] / valid[total_col]).mean()
            print(f"  {proj_name:12s}: {cr:.3f} ({cr*100:.1f}%)")
        else:
            print(f"  {proj_name:12s}: Insufficient data")
    else:
        print(f"  {proj_name:12s}: Required columns missing")


In [ ]:
# ── Current Story Point Distribution — Histograms ────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("Current Story Point Distribution per Project (Issues Summary)",
             fontsize=14, fontweight="bold")

issue_summaries = {
    "Mesos":     MESO_Issue_Summary_370,
    "Spring XD": Spring_XD_Issues_Summary_2861,
    "Aurora":    Aurora_Issues_summery_568,
    "Usergrid":  Usergrid_Issues_Summary_929,
}

for ax, (proj_name, iss_df) in zip(axes.flat, issue_summaries.items()):
    if iss_df is None:
        ax.set_title(f"{proj_name} — NOT LOADED")
        continue
    if "currentStoryPoint" not in iss_df.columns:
        ax.set_title(f"{proj_name} — No story point col")
        continue
    sp = pd.to_numeric(iss_df["currentStoryPoint"], errors="coerce").dropna()
    sp = sp[sp >= 0].clip(upper=sp.quantile(0.97))
    color = project_colors[proj_name]
    ax.hist(sp, bins=25, color=color, alpha=0.8, edgecolor="white")
    ax.axvline(sp.mean(), color="black", linestyle="--", linewidth=1.5,
               label=f"Mean: {sp.mean():.1f}")
    ax.axvline(sp.median(), color="red", linestyle=":", linewidth=1.5,
               label=f"Median: {sp.median():.1f}")
    ax.set_title(f"{proj_name} — Story Points", fontsize=11, fontweight="bold")
    ax.set_xlabel("Current Story Points")
    ax.set_ylabel("Frequency")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "story_point_distributions.png", bbox_inches="tight")
plt.show()


## Cell 5 — Exploratory Data Analysis: Project Risk Dataset

Deep-dive into the `project_risk_raw_dataset` (4,000 records, 48 features).
- Risk_Level class distribution (pie chart + count plot)
- Top 10 features correlated with Risk_Level
- Complexity_Score vs Schedule_Pressure scatter coloured by Risk_Level
- Team_Turnover_Rate by Risk_Level box plot
- Null value audit


In [ ]:
if project_risk_raw_dataset is not None:
    prisk = project_risk_raw_dataset.copy()
    print(f"Shape: {prisk.shape}")
    print(f"\nRisk_Level distribution:")
    print(prisk["Risk_Level"].value_counts())
    print(f"\nNull counts (columns with nulls > 0):")
    nulls = prisk.isnull().sum()
    print(nulls[nulls > 0])
else:
    print("⚠️  project_risk_raw_dataset not loaded.")


In [ ]:
# ── Risk_Level Distribution — Pie + Count Plot ───────────────────────────────
if project_risk_raw_dataset is not None:
    prisk = project_risk_raw_dataset.copy()
    risk_order = ["Low", "Medium", "High", "Critical"]
    risk_colors = {"Low": "#2ECC71", "Medium": "#F0A500",
                   "High": "#E67E22", "Critical": "#E74C3C"}

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle("Project Risk Dataset — Risk Level Distribution",
                 fontsize=13, fontweight="bold")

    # Pie chart
    counts = prisk["Risk_Level"].value_counts()
    pie_labels = [l for l in risk_order if l in counts.index]
    pie_values = [counts[l] for l in pie_labels]
    pie_colors = [risk_colors[l] for l in pie_labels]
    axes[0].pie(pie_values, labels=pie_labels, colors=pie_colors,
                autopct="%1.1f%%", startangle=90,
                wedgeprops={"edgecolor": "white", "linewidth": 2})
    axes[0].set_title("Proportion of Risk Levels", fontsize=11)

    # Count plot
    plot_order = [l for l in risk_order if l in counts.index]
    bar_colors = [risk_colors[l] for l in plot_order]
    axes[1].bar(plot_order, [counts[l] for l in plot_order],
                color=bar_colors, edgecolor="white", linewidth=1.5)
    axes[1].set_title("Count by Risk Level", fontsize=11)
    axes[1].set_xlabel("Risk Level")
    axes[1].set_ylabel("Count")
    for i, (l, v) in enumerate(zip(plot_order, [counts[l] for l in plot_order])):
        axes[1].text(i, v + 5, str(v), ha="center", fontsize=10, fontweight="bold")
    axes[1].grid(True, axis="y", alpha=0.3)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "risk_level_distribution.png", bbox_inches="tight")
    plt.show()


In [ ]:
# ── Top 10 Correlations with Risk_Level ──────────────────────────────────────
if project_risk_raw_dataset is not None:
    prisk = project_risk_raw_dataset.copy()
    risk_map = {"Low": 0, "Medium": 1, "High": 2, "Critical": 3}
    prisk["Risk_Level_enc"] = prisk["Risk_Level"].map(risk_map)

    # Encode categorical columns for correlation
    prisk_enc = prisk.copy()
    for col in prisk_enc.select_dtypes(include=["object"]).columns:
        if col != "Risk_Level":
            try:
                prisk_enc[col] = LabelEncoder().fit_transform(
                    prisk_enc[col].astype(str))
            except Exception:
                prisk_enc.drop(columns=[col], inplace=True)

    numeric = prisk_enc.select_dtypes(include=[np.number])
    if "Risk_Level_enc" in numeric.columns:
        corr_with_risk = (numeric.corr()["Risk_Level_enc"]
                          .drop("Risk_Level_enc")
                          .abs()
                          .sort_values(ascending=False)
                          .head(10))

        fig, ax = plt.subplots(figsize=(10, 6))
        colors_bar = ["#E74C3C" if v > 0.3 else "#3498DB" if v > 0.15
                      else "#95A5A6" for v in corr_with_risk.values]
        ax.barh(corr_with_risk.index[::-1], corr_with_risk.values[::-1],
                color=colors_bar[::-1], edgecolor="white")
        ax.set_title("Top 10 Features by Absolute Pearson Correlation with Risk_Level",
                     fontsize=12, fontweight="bold")
        ax.set_xlabel("|Pearson Correlation|")
        ax.axvline(0.3, color="red", linestyle="--", alpha=0.5, label="r=0.3")
        ax.axvline(0.15, color="orange", linestyle="--", alpha=0.5, label="r=0.15")
        ax.legend()
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / "risk_top_correlations.png", bbox_inches="tight")
        plt.show()
        print("Top 10 correlations with Risk_Level:")
        print(corr_with_risk.round(4))


In [ ]:
# ── Complexity_Score vs Schedule_Pressure coloured by Risk_Level ─────────────
if project_risk_raw_dataset is not None:
    prisk = project_risk_raw_dataset.copy()
    if "Complexity_Score" in prisk.columns and "Schedule_Pressure" in prisk.columns:
        prisk["Complexity_Score"] = pd.to_numeric(prisk["Complexity_Score"], errors="coerce")
        prisk["Schedule_Pressure"] = pd.to_numeric(prisk["Schedule_Pressure"], errors="coerce")
        prisk = prisk.dropna(subset=["Complexity_Score", "Schedule_Pressure"])

        risk_order = ["Low", "Medium", "High", "Critical"]
        risk_colors = {"Low": "#2ECC71", "Medium": "#F0A500",
                       "High": "#E67E22", "Critical": "#E74C3C"}
        risk_markers = {"Low": "o", "Medium": "s", "High": "^", "Critical": "D"}

        fig, ax = plt.subplots(figsize=(12, 7))
        for level in risk_order:
            mask = prisk["Risk_Level"] == level
            subset = prisk[mask]
            if len(subset) == 0:
                continue
            ax.scatter(subset["Complexity_Score"], subset["Schedule_Pressure"],
                       c=risk_colors[level], marker=risk_markers[level],
                       label=level, alpha=0.6, s=25, edgecolors="none")

        ax.set_title("Complexity Score vs Schedule Pressure by Risk Level",
                     fontsize=12, fontweight="bold")
        ax.set_xlabel("Complexity Score")
        ax.set_ylabel("Schedule Pressure")
        ax.legend(title="Risk Level", fontsize=9)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / "complexity_vs_schedule.png", bbox_inches="tight")
        plt.show()


In [ ]:
# ── Team_Turnover_Rate by Risk_Level — Box Plot ───────────────────────────────
if project_risk_raw_dataset is not None:
    prisk = project_risk_raw_dataset.copy()
    if "Team_Turnover_Rate" in prisk.columns:
        prisk["Team_Turnover_Rate"] = pd.to_numeric(prisk["Team_Turnover_Rate"], errors="coerce")
        risk_order = ["Low", "Medium", "High", "Critical"]
        risk_colors_list = ["#2ECC71", "#F0A500", "#E67E22", "#E74C3C"]
        groups = [prisk.loc[prisk["Risk_Level"] == lv, "Team_Turnover_Rate"].dropna().values
                  for lv in risk_order if lv in prisk["Risk_Level"].unique()]
        valid_labels = [lv for lv in risk_order if lv in prisk["Risk_Level"].unique()]

        fig, ax = plt.subplots(figsize=(10, 6))
        bp = ax.boxplot(groups, labels=valid_labels, patch_artist=True,
                        notch=False, vert=True)
        colors_box = [risk_colors_list[risk_order.index(l)] for l in valid_labels]
        for patch, color in zip(bp["boxes"], colors_box):
            patch.set_facecolor(color)
            patch.set_alpha(0.75)
        ax.set_title("Team Turnover Rate Distribution by Risk Level",
                     fontsize=12, fontweight="bold")
        ax.set_xlabel("Risk Level")
        ax.set_ylabel("Team Turnover Rate")
        ax.grid(True, axis="y", alpha=0.3)
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / "turnover_by_risk.png", bbox_inches="tight")
        plt.show()


## Cell 6 — Exploratory Data Analysis: Agile Projects Outcome Dataset

Analyse `Agile_Projects_Dataset` (200 records) covering project success outcomes.
- Pearson correlation heatmap
- Project Success distribution
- Feature-vs-success grouped box plots
- Descriptive statistics


In [ ]:
if Agile_Projects_Dataset is not None:
    agile_pd = Agile_Projects_Dataset.copy()
    print(f"Shape: {agile_pd.shape}")
    print("\nColumns:", agile_pd.columns.tolist())
    print("\nDescriptive Statistics:")
    print(agile_pd.describe().round(3))


In [ ]:
# ── Pearson Correlation Heatmap ───────────────────────────────────────────────
if Agile_Projects_Dataset is not None:
    agile_pd = Agile_Projects_Dataset.copy()
    numeric_cols = agile_pd.select_dtypes(include=[np.number]).columns.tolist()
    if len(numeric_cols) >= 2:
        corr = agile_pd[numeric_cols].corr()
        fig, ax = plt.subplots(figsize=(10, 8))
        sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm",
                    linewidths=0.5, ax=ax, annot_kws={"size": 8},
                    vmin=-1, vmax=1, center=0)
        ax.set_title("Agile Projects Dataset — Pearson Correlation Heatmap",
                     fontsize=12, fontweight="bold")
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / "agile_projects_correlation.png", bbox_inches="tight")
        plt.show()


In [ ]:
# ── Project Success Distribution ─────────────────────────────────────────────
if Agile_Projects_Dataset is not None:
    agile_pd = Agile_Projects_Dataset.copy()
    success_col = None
    for c in agile_pd.columns:
        if "success" in c.lower():
            success_col = c
            break

    if success_col:
        fig, axes = plt.subplots(1, 2, figsize=(13, 5))
        success_vals = pd.to_numeric(agile_pd[success_col], errors="coerce").dropna()

        axes[0].hist(success_vals, bins=20, color="#3498DB", edgecolor="white", alpha=0.85)
        axes[0].axvline(success_vals.mean(), color="red", linestyle="--",
                        label=f"Mean: {success_vals.mean():.2f}")
        axes[0].set_title(f"Distribution of '{success_col}'", fontweight="bold")
        axes[0].set_xlabel(success_col)
        axes[0].set_ylabel("Frequency")
        axes[0].legend()

        if agile_pd[success_col].nunique() <= 10:
            counts = agile_pd[success_col].value_counts().sort_index()
            axes[1].bar(counts.index.astype(str), counts.values,
                        color="#2ECC71", edgecolor="white")
            axes[1].set_title(f"'{success_col}' Count by Category", fontweight="bold")
            axes[1].set_xlabel(success_col)
            axes[1].set_ylabel("Count")
        else:
            axes[1].boxplot(success_vals, patch_artist=True,
                            boxprops={"facecolor": "#3498DB", "alpha": 0.7})
            axes[1].set_title(f"'{success_col}' Box Plot", fontweight="bold")

        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / "agile_success_distribution.png", bbox_inches="tight")
        plt.show()


In [ ]:
# ── Feature vs Project Success — Grouped Box Plots ───────────────────────────
if Agile_Projects_Dataset is not None:
    agile_pd = Agile_Projects_Dataset.copy()
    success_col = None
    for c in agile_pd.columns:
        if "success" in c.lower():
            success_col = c
            break

    feature_cols = ["Agile Effectiveness", "Risk Mitigation", "Management Satisfaction",
                    "Supply Chain Improvement", "Time Efficiency", "Cost Savings (%)"]
    feature_cols = [f for f in feature_cols if f in agile_pd.columns]

    if success_col and feature_cols:
        agile_pd[success_col] = pd.to_numeric(agile_pd[success_col], errors="coerce")
        # Bin success into Low/High for grouping if continuous
        if agile_pd[success_col].nunique() > 5:
            median_val = agile_pd[success_col].median()
            agile_pd["Success_Group"] = agile_pd[success_col].apply(
                lambda x: "High" if x >= median_val else "Low" if pd.notna(x) else np.nan)
        else:
            agile_pd["Success_Group"] = agile_pd[success_col].astype(str)

        n_features = len(feature_cols)
        ncols = 3
        nrows = (n_features + ncols - 1) // ncols
        fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 4))
        axes = axes.flat

        for ax, feat in zip(axes, feature_cols):
            groups_data = {}
            for grp in agile_pd["Success_Group"].dropna().unique():
                vals = pd.to_numeric(
                    agile_pd.loc[agile_pd["Success_Group"] == grp, feat],
                    errors="coerce").dropna().values
                if len(vals) > 0:
                    groups_data[grp] = vals

            if groups_data:
                bp = ax.boxplot(list(groups_data.values()),
                                labels=list(groups_data.keys()),
                                patch_artist=True, notch=False)
                palette = ["#3498DB", "#E74C3C", "#2ECC71", "#F0A500"]
                for patch, c in zip(bp["boxes"], palette):
                    patch.set_facecolor(c)
                    patch.set_alpha(0.7)
                ax.set_title(f"{feat}", fontsize=10, fontweight="bold")
                ax.set_ylabel("Score")
                ax.grid(True, axis="y", alpha=0.3)

        # Hide unused subplots
        for ax in list(axes)[n_features:]:
            ax.set_visible(False)

        fig.suptitle("Feature Distributions by Project Success Group",
                     fontsize=13, fontweight="bold", y=1.01)
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / "agile_features_vs_success.png", bbox_inches="tight")
        plt.show()


## Cell 7 — Feature Engineering

Implement all engineered features for the Agile sprint datasets as specified in the dissertation artefact:

| Feature | Formula |
|---|---|
| `velocity` | `completedIssuesEstimateSum` (or `completedIssuesCount`) |
| `velocity_variance` | Rolling std of velocity (window=3) |
| `completion_rate` | `completedIssuesCount / totalNumberOfIssues` |
| `backlog_growth` | Count of `issueKeysAddedDuringSprint` |
| `punt_rate` | `puntedIssues / totalNumberOfIssues` |
| `schedule_variance` | `(sprintCompleteDate − sprintEndDate)` in days |
| `defect_density` | `branchCount / loc` (from NASA datasets) |
| `sprint_risk_label` | 1 if completion_rate<0.6 OR punt_rate>0.3 OR schedule_variance>2 |
| `avg_story_points` | Mean `currentStoryPoint` per sprint (from Issues Summary) |
| `story_point_delta` | Mean `(currentStoryPoint − initialStoryPoint)` per sprint |
| `not_completed_ratio` | Count(not completed) / total issues per sprint |
| `assignee_diversity` | Count of unique assignees per sprint |


In [ ]:
# ── Helper functions ─────────────────────────────────────────────────────────

def parse_date_safe(s):
    """Parse a date string to datetime, returning NaT on failure."""
    if pd.isna(s):
        return pd.NaT
    try:
        return pd.to_datetime(str(s), infer_datetime_format=True, errors="coerce")
    except Exception:
        return pd.NaT

def count_issue_keys(val):
    """Count number of issue keys in a cell (JSON array, comma-sep, or numeric)."""
    if pd.isna(val) or val == "" or val == "[]":
        return 0
    if isinstance(val, (int, float)):
        return int(val)
    if isinstance(val, str):
        try:
            parsed = json.loads(val)
            return len(parsed)
        except Exception:
            pass
        return len([x for x in re.split(r"[,;|]", val) if x.strip()])
    return 0

def safe_num(df, col):
    """Convert column to numeric, filling NaN with 0."""
    return pd.to_numeric(df[col], errors="coerce").fillna(0) if col in df.columns else pd.Series(0, index=df.index)


In [ ]:
# ── Core sprint feature engineering function ─────────────────────────────────

def engineer_sprint_features(sprints_df, issues_summary_df, project_name="Unknown"):
    """
    Compute sprint-level risk proxy features from a Sprint dataframe
    and its corresponding Issues Summary dataframe.
    Returns an enriched dataframe with sprint_risk_label.
    """
    if sprints_df is None:
        print(f"  ⚠️  {project_name}: Sprint dataframe is None — skipping.")
        return pd.DataFrame()

    df = sprints_df.copy().reset_index(drop=True)
    df["project_name"] = project_name

    # ── Normalise total issues column ────────────────────────────────────────
    total_col = None
    for c in ["totalNumberOfIssues", "total"]:
        if c in df.columns:
            total_col = c
            break
    if total_col is None:
        df["_total"] = 0
        total_col = "_total"
    df["_total_issues"] = safe_num(df, total_col).replace(0, np.nan)

    # ── velocity ──────────────────────────────────────────────────────────────
    if "completedIssuesEstimateSum" in df.columns:
        df["velocity"] = safe_num(df, "completedIssuesEstimateSum")
    elif "completedIssuesCount" in df.columns:
        df["velocity"] = safe_num(df, "completedIssuesCount")
    else:
        df["velocity"] = 0.0

    # ── velocity_variance (rolling std, window=3) ─────────────────────────────
    df["velocity_variance"] = df["velocity"].rolling(window=3, min_periods=1).std().fillna(0)

    # ── completion_rate ───────────────────────────────────────────────────────
    completed = safe_num(df, "completedIssuesCount")
    df["completion_rate"] = np.where(
        df["_total_issues"].notna() & (df["_total_issues"] > 0),
        completed / df["_total_issues"],
        np.nan
    )
    df["completion_rate"] = df["completion_rate"].clip(0, 1).fillna(0)

    # ── backlog_growth ────────────────────────────────────────────────────────
    if "issueKeysAddedDuringSprint" in df.columns:
        df["backlog_growth"] = df["issueKeysAddedDuringSprint"].apply(count_issue_keys)
    else:
        df["backlog_growth"] = 0

    # ── punt_rate ─────────────────────────────────────────────────────────────
    punted = safe_num(df, "puntedIssues")
    df["punt_rate"] = np.where(
        df["_total_issues"].notna() & (df["_total_issues"] > 0),
        punted / df["_total_issues"],
        0.0
    )
    df["punt_rate"] = df["punt_rate"].clip(0, 1)

    # ── schedule_variance (positive = late) ───────────────────────────────────
    if "sprintCompleteDate" in df.columns and "sprintEndDate" in df.columns:
        complete_dt = df["sprintCompleteDate"].apply(parse_date_safe)
        end_dt      = df["sprintEndDate"].apply(parse_date_safe)
        diff = (complete_dt - end_dt).dt.total_seconds() / 86400.0
        df["schedule_variance"] = diff.fillna(0)
    else:
        df["schedule_variance"] = 0.0

    # ── sprint_risk_label ─────────────────────────────────────────────────────
    df["sprint_risk_label"] = (
        (df["completion_rate"] < 0.6) |
        (df["punt_rate"] > 0.3) |
        (df["schedule_variance"] > 2)
    ).astype(int)

    # ── Merge with Issues Summary for per-sprint aggregates ──────────────────
    if issues_summary_df is not None and "sprintId" in df.columns:
        iss = issues_summary_df.copy()
        if "sprintId" not in iss.columns:
            print(f"  ⚠️  {project_name}: No sprintId in issues summary.")
        else:
            iss["currentStoryPoint"]  = pd.to_numeric(iss.get("currentStoryPoint",  np.nan), errors="coerce")
            iss["initialStoryPoint"]  = pd.to_numeric(iss.get("initialStoryPoint",  np.nan), errors="coerce")
            iss["story_point_delta_"]  = iss["currentStoryPoint"] - iss["initialStoryPoint"]

            # Status: not completed
            not_done_statuses = {"NotCompletedWithinSprint", "Open", "In Progress",
                                 "Reopened", "TODO", "Backlog"}
            if "status" in iss.columns:
                iss["not_completed_flag"] = iss["status"].apply(
                    lambda s: 1 if str(s).strip() in not_done_statuses else 0)
            else:
                iss["not_completed_flag"] = 0

            agg = iss.groupby("sprintId").agg(
                avg_story_points    = ("currentStoryPoint", "mean"),
                story_point_delta   = ("story_point_delta_", "mean"),
                not_completed_count = ("not_completed_flag", "sum"),
                total_issues_iss    = ("currentStoryPoint", "count"),
                assignee_diversity  = ("assignee", lambda x: x.nunique()) if "assignee" in iss.columns else ("currentStoryPoint", "count"),
            ).reset_index()

            agg["not_completed_ratio"] = np.where(
                agg["total_issues_iss"] > 0,
                agg["not_completed_count"] / agg["total_issues_iss"],
                0.0
            )
            df = df.merge(agg[["sprintId", "avg_story_points", "story_point_delta",
                                "not_completed_ratio", "assignee_diversity"]],
                          on="sprintId", how="left")
    # Fill missing issue-summary cols
    for col in ["avg_story_points", "story_point_delta", "not_completed_ratio", "assignee_diversity"]:
        if col not in df.columns:
            df[col] = 0.0
        else:
            df[col] = df[col].fillna(0)

    # ── Drop helper col ───────────────────────────────────────────────────────
    df.drop(columns=["_total_issues"], inplace=True, errors="ignore")

    return df


In [ ]:
# ── Engineer features for all 4 projects ─────────────────────────────────────
print("Engineering sprint features for all projects...\n")

mesos_engineered    = engineer_sprint_features(MESO_Sprint_96,
                                               MESO_Issue_Summary_370,
                                               "Mesos")
springxd_engineered = engineer_sprint_features(Spring_XD_Sprints_67,
                                               Spring_XD_Issues_Summary_2861,
                                               "Spring_XD")
aurora_engineered   = engineer_sprint_features(Aurora_Sprints_41,
                                               Aurora_Issues_summery_568,
                                               "Aurora")
usergrid_engineered = engineer_sprint_features(Usergrid_Sprints_36,
                                               Usergrid_Issues_Summary_929,
                                               "Usergrid")

# ── Combine all 4 ─────────────────────────────────────────────────────────────
agile_merged_df = pd.concat(
    [df for df in [mesos_engineered, springxd_engineered,
                   aurora_engineered, usergrid_engineered]
     if df is not None and len(df) > 0],
    ignore_index=True
)

print(f"Combined agile_merged_df shape: {agile_merged_df.shape}")
print("\nEngineered feature columns:")
eng_cols = ["velocity", "velocity_variance", "completion_rate", "backlog_growth",
            "punt_rate", "schedule_variance", "sprint_risk_label",
            "avg_story_points", "story_point_delta", "not_completed_ratio",
            "assignee_diversity", "project_name"]
print([c for c in eng_cols if c in agile_merged_df.columns])
print("\nFirst 5 rows:")
display_cols = [c for c in eng_cols if c in agile_merged_df.columns]
print(agile_merged_df[display_cols].head())


In [ ]:
# ── sprint_risk_label value counts per project ───────────────────────────────
print("\n📊 sprint_risk_label Distribution per Project:")
print("-" * 50)
if "sprint_risk_label" in agile_merged_df.columns and "project_name" in agile_merged_df.columns:
    pivot = agile_merged_df.groupby(["project_name", "sprint_risk_label"]).size().unstack(fill_value=0)
    pivot.columns = [f"Label={c}" for c in pivot.columns]
    pivot["Total"] = pivot.sum(axis=1)
    print(pivot)
    print("\nOverall:")
    print(agile_merged_df["sprint_risk_label"].value_counts())
else:
    print("Columns missing.")

# ── Defect density from NASA datasets ─────────────────────────────────────────
print("\n📐 Computing defect_density from NASA datasets (branchCount / loc)...")
nasa_combined_parts = []
for name, df in nasa_datasets.items():
    if df is None:
        continue
    d = df.copy()
    if "loc" in d.columns and "branchCount" in d.columns:
        d["defect_density"] = np.where(
            pd.to_numeric(d["loc"], errors="coerce") > 0,
            pd.to_numeric(d["branchCount"], errors="coerce") /
            pd.to_numeric(d["loc"], errors="coerce"),
            0.0
        )
        print(f"  {name}: defect_density mean = {d['defect_density'].mean():.4f}")
    else:
        d["defect_density"] = 0.0
    nasa_combined_parts.append(d)

print("\n✅ Feature engineering complete.")


## Cell 8 — Data Preprocessing & Merging

Prepare each dataset group for model training:
1. **NASA datasets**: concatenate, standardise column names, encode kc2 target
2. **Project Risk dataset**: impute nulls, one-hot encode, encode `Risk_Level`
3. **Agile sprint features**: already engineered above
4. **Merge** into a single `merged_train_data` dataframe
5. Save `merged_train_data.csv` to `/content/`


In [ ]:
# ── Prepare NASA Datasets ─────────────────────────────────────────────────────

def prepare_nasa_datasets(datasets_dict):
    """
    Concatenate cm1, pc1, jm1, kc1, kc2.
    Standardises column names, encodes kc2 target, adds dataset_source.
    Returns cleaned NASA defect dataframe.
    """
    nasa_parts = []
    for name in ["cm1", "pc1", "jm1", "kc1", "kc2"]:
        df = datasets_dict.get(name)
        if df is None:
            print(f"  ⚠️  {name}: skipped (not loaded)")
            continue
        d = df.copy()
        d["dataset_source"] = name

        # Normalise target to 'defects' as int 0/1
        if name == "kc2":
            if "problems" in d.columns:
                d["defects"] = (d["problems"].astype(str).str.lower()
                                .map({"yes": 1, "no": 0, "true": 1, "false": 0})
                                .fillna(0).astype(int))
                d.drop(columns=["problems"], inplace=True)
        else:
            if "defects" in d.columns:
                t = d["defects"]
                if t.dtype == bool:
                    d["defects"] = t.astype(int)
                else:
                    d["defects"] = (t.astype(str).str.lower()
                                    .map({"true": 1, "false": 0, "1": 1, "0": 0})
                                    .fillna(0).astype(int))

        # Standardise iv(G) / iv(g) column name
        col_map = {}
        for c in d.columns:
            if c.lower() == "iv(g)":
                col_map[c] = "iv(g)"
        if col_map:
            d.rename(columns=col_map, inplace=True)

        # Add defect_density
        if "loc" in d.columns and "branchCount" in d.columns:
            loc_n  = pd.to_numeric(d["loc"], errors="coerce")
            bc_n   = pd.to_numeric(d["branchCount"], errors="coerce")
            d["defect_density"] = np.where(loc_n > 0, bc_n / loc_n, 0.0)
        else:
            d["defect_density"] = 0.0

        nasa_parts.append(d)

    if not nasa_parts:
        return pd.DataFrame()

    nasa_df = pd.concat(nasa_parts, ignore_index=True)
    # Fill numeric nulls with median
    for col in nasa_df.select_dtypes(include=[np.number]).columns:
        nasa_df[col] = nasa_df[col].fillna(nasa_df[col].median())
    print(f"NASA combined shape: {nasa_df.shape}")
    return nasa_df

nasa_combined = prepare_nasa_datasets(nasa_datasets)
print(f"\nNASA target distribution:")
if "defects" in nasa_combined.columns:
    print(nasa_combined["defects"].value_counts())


In [ ]:
# ── Prepare Project Risk Dataset ─────────────────────────────────────────────

def prepare_risk_dataset(prisk_df):
    """
    Impute nulls, one-hot encode categoricals, encode Risk_Level.
    Returns cleaned risk dataframe with Risk_Level_enc column.
    """
    if prisk_df is None:
        return pd.DataFrame()

    d = prisk_df.copy()

    # Map Risk_Level to integers
    risk_map = {"Low": 0, "Medium": 1, "High": 2, "Critical": 3}
    if "Risk_Level" in d.columns:
        d["Risk_Level_enc"] = d["Risk_Level"].map(risk_map)
    else:
        d["Risk_Level_enc"] = 0

    # Impute numeric columns with median
    num_cols = d.select_dtypes(include=[np.number]).columns.tolist()
    for col in num_cols:
        if d[col].isnull().sum() > 0:
            d[col] = d[col].fillna(d[col].median())

    # Impute categorical columns with mode
    cat_cols = d.select_dtypes(include=["object"]).columns.tolist()
    cat_cols = [c for c in cat_cols if c not in ["Risk_Level"]]
    for col in cat_cols:
        if d[col].isnull().sum() > 0:
            mode_val = d[col].mode()
            d[col] = d[col].fillna(mode_val[0] if len(mode_val) > 0 else "Unknown")

    # One-hot encode categorical columns (drop first to avoid multicollinearity)
    if cat_cols:
        d = pd.get_dummies(d, columns=cat_cols, drop_first=True, dtype=int)

    print(f"Risk dataset shape after encoding: {d.shape}")
    return d

risk_prepared = prepare_risk_dataset(project_risk_raw_dataset)
if "Risk_Level_enc" in risk_prepared.columns:
    print("\nRisk_Level_enc distribution:")
    print(risk_prepared["Risk_Level_enc"].value_counts().sort_index())


In [ ]:
# ── Build Merged Training Dataset ────────────────────────────────────────────

SPRINT_FEATURE_COLS = [
    "velocity", "velocity_variance", "completion_rate", "backlog_growth",
    "punt_rate", "schedule_variance", "avg_story_points", "story_point_delta",
    "not_completed_ratio", "assignee_diversity"
]
NASA_FEATURE_COLS = [
    "loc", "v(g)", "ev(g)", "iv(g)", "n", "v", "l", "d", "i", "e",
    "b", "t", "lOCode", "lOComment", "lOBlank", "uniq_Op", "uniq_Opnd",
    "total_Op", "total_Opnd", "branchCount", "defect_density"
]
TARGET_COL = "sprint_risk_label"

def build_merged_training_data(agile_df, nasa_df, risk_df, agile_projects_df, output_dir=None):
    """
    Select harmonised feature subsets, align schemas, and concatenate into merged_train_data.
    Primary target: sprint_risk_label (binary 0/1).
    """
    all_parts = []

    # ── 1. Agile sprint data ──────────────────────────────────────────────────
    if agile_df is not None and len(agile_df) > 0:
        sprint_available = [c for c in SPRINT_FEATURE_COLS if c in agile_df.columns]
        if "sprint_risk_label" in agile_df.columns:
            part_sprint = agile_df[sprint_available + ["sprint_risk_label"]].copy()
            part_sprint[TARGET_COL] = part_sprint["sprint_risk_label"]
            # Pad missing sprint features with 0
            for c in SPRINT_FEATURE_COLS:
                if c not in part_sprint.columns:
                    part_sprint[c] = 0.0
            # Pad missing NASA features with 0
            for c in NASA_FEATURE_COLS:
                part_sprint[c] = 0.0
            all_parts.append(part_sprint)
            print(f"Agile sprint rows: {len(part_sprint)}")

    # ── 2. NASA data ──────────────────────────────────────────────────────────
    if nasa_df is not None and len(nasa_df) > 0 and "defects" in nasa_df.columns:
        nasa_avail = [c for c in NASA_FEATURE_COLS if c in nasa_df.columns]
        part_nasa = nasa_df[nasa_avail + ["defects"]].copy()
        part_nasa[TARGET_COL] = part_nasa["defects"].astype(int)
        for c in SPRINT_FEATURE_COLS:
            part_nasa[c] = 0.0
        for c in NASA_FEATURE_COLS:
            if c not in part_nasa.columns:
                part_nasa[c] = 0.0
        all_parts.append(part_nasa)
        print(f"NASA rows: {len(part_nasa)}")

    # ── 3. Risk dataset (map High/Critical → 1, else → 0 for binary label) ───
    if risk_df is not None and len(risk_df) > 0 and "Risk_Level_enc" in risk_df.columns:
        part_risk = risk_df.copy()
        # Binary label: High(2) or Critical(3) = 1, else = 0
        part_risk[TARGET_COL] = (part_risk["Risk_Level_enc"] >= 2).astype(int)
        for c in SPRINT_FEATURE_COLS:
            if c not in part_risk.columns:
                part_risk[c] = 0.0
        for c in NASA_FEATURE_COLS:
            if c not in part_risk.columns:
                part_risk[c] = 0.0
        # Keep only the feature columns + target
        keep_cols = SPRINT_FEATURE_COLS + NASA_FEATURE_COLS + [TARGET_COL]
        keep_cols = [c for c in keep_cols if c in part_risk.columns]
        all_parts.append(part_risk[keep_cols])
        print(f"Risk dataset rows: {len(part_risk)}")

    if not all_parts:
        print("⚠️  No data available to merge!")
        return pd.DataFrame()

    # ── Align all schemas ─────────────────────────────────────────────────────
    all_feature_cols = list(set(SPRINT_FEATURE_COLS + NASA_FEATURE_COLS))
    merged = pd.concat(all_parts, ignore_index=True)

    # Ensure all required columns exist
    for c in all_feature_cols:
        if c not in merged.columns:
            merged[c] = 0.0

    # Fill any remaining NaN
    for c in all_feature_cols:
        merged[c] = pd.to_numeric(merged[c], errors="coerce").fillna(0)
    merged[TARGET_COL] = merged[TARGET_COL].fillna(0).astype(int)

    # Keep only feature + target columns
    final_cols = all_feature_cols + [TARGET_COL]
    final_cols = [c for c in final_cols if c in merged.columns]
    merged = merged[final_cols]

    print(f"\nmerged_train_data shape: {merged.shape}")
    print(f"Columns: {merged.columns.tolist()}")
    print(f"\n{TARGET_COL} distribution:")
    print(merged[TARGET_COL].value_counts())

    if output_dir is not None:
        save_path = pathlib.Path(output_dir) / "merged_train_data.csv"
        merged.to_csv(save_path, index=False)
        print(f"\n✅ Saved to {save_path}")

    return merged

merged_train_data = build_merged_training_data(
    agile_df=agile_merged_df,
    nasa_df=nasa_combined,
    risk_df=risk_prepared,
    agile_projects_df=Agile_Projects_Dataset,
    output_dir=str(CONTENT_DIR)
)

In [ ]:
# ── Final class distribution bar chart ────────────────────────────────────────
if len(merged_train_data) > 0 and TARGET_COL in merged_train_data.columns:
    fig, ax = plt.subplots(figsize=(7, 4))
    counts = merged_train_data[TARGET_COL].value_counts().sort_index()
    labels = ["No Risk (0)", "At Risk (1)"][:len(counts)]
    bar_colors = ["#2ECC71", "#E74C3C"][:len(counts)]
    ax.bar(labels, counts.values, color=bar_colors, edgecolor="white", linewidth=1.5)
    ax.set_title(f"merged_train_data — Target Class Distribution ({TARGET_COL})",
                 fontsize=12, fontweight="bold")
    ax.set_ylabel("Count")
    for i, v in enumerate(counts.values):
        ax.text(i, v + 50, f"{v:,}\n({v/len(merged_train_data)*100:.1f}%)",
                ha="center", fontsize=10, fontweight="bold")
    ax.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "merged_class_distribution.png", bbox_inches="tight")
    plt.show()
    print(f"\nTotal records: {len(merged_train_data):,}")
    print(f"Total features: {merged_train_data.shape[1]-1}")
    print(f"Null values: {merged_train_data.isnull().sum().sum()}")


## Cell 9 — Preliminary Inferential Analysis: Pearson Correlation

Compute full Pearson correlation matrix on `merged_train_data` to identify which features
are most predictive of the sprint risk label.
- Full annotated heatmap
- Top 15 features by absolute correlation with target
- scipy `pearsonr` for top 5 pairs with r-value, p-value, and significance interpretation


In [ ]:
# ── Full Pearson correlation matrix ──────────────────────────────────────────
if len(merged_train_data) > 0:
    numeric_merged = merged_train_data.select_dtypes(include=[np.number])
    corr_full = numeric_merged.corr(method="pearson")

    fig, ax = plt.subplots(figsize=(16, 13))
    mask = np.triu(np.ones_like(corr_full, dtype=bool))
    sns.heatmap(corr_full, mask=mask, annot=True, fmt=".2f",
                cmap="RdYlGn", linewidths=0.4, ax=ax,
                annot_kws={"size": 7}, vmin=-1, vmax=1, center=0)
    ax.set_title("merged_train_data — Full Pearson Correlation Matrix",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "merged_pearson_heatmap.png", bbox_inches="tight")
    plt.show()


In [ ]:
# ── Top 15 features correlated with target ───────────────────────────────────
if len(merged_train_data) > 0 and TARGET_COL in corr_full.columns:
    top15 = (corr_full[TARGET_COL].drop(TARGET_COL)
             .abs().sort_values(ascending=False).head(15))

    fig, ax = plt.subplots(figsize=(10, 6))
    colors_t = ["#E74C3C" if v > 0.3 else "#F0A500" if v > 0.15
                else "#3498DB" for v in top15.values]
    ax.barh(top15.index[::-1], top15.values[::-1], color=colors_t[::-1])
    ax.set_title(f"Top 15 Features by |Pearson r| with '{TARGET_COL}'",
                 fontsize=12, fontweight="bold")
    ax.set_xlabel("|Pearson Correlation Coefficient|")
    ax.axvline(0.3, color="red", linestyle="--", alpha=0.6, label="Strong (r=0.3)")
    ax.axvline(0.1, color="orange", linestyle="--", alpha=0.6, label="Weak (r=0.1)")
    ax.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "top15_correlations.png", bbox_inches="tight")
    plt.show()

    print("\n📊 Top 15 Features by Correlation with Target:")
    print(top15.round(4).to_string())


In [ ]:
# ── scipy pearsonr for top 5 pairs ───────────────────────────────────────────
if len(merged_train_data) > 0 and TARGET_COL in corr_full.columns:
    top5_features = (corr_full[TARGET_COL].drop(TARGET_COL)
                     .abs().sort_values(ascending=False).head(5).index.tolist())

    print("\n📐 Pearson Correlation (r) and Significance (p) — Top 5 Features vs Target")
    print("=" * 75)
    y_vals = merged_train_data[TARGET_COL].values
    for feat in top5_features:
        if feat in merged_train_data.columns:
            x_vals = merged_train_data[feat].values
            # Remove NaN pairs
            mask_valid = (~np.isnan(x_vals)) & (~np.isnan(y_vals))
            if mask_valid.sum() < 3:
                continue
            r, p = pearsonr(x_vals[mask_valid], y_vals[mask_valid])
            stars = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
            interp = ("Strong positive" if r > 0.3
                      else "Moderate positive" if r > 0.1
                      else "Strong negative" if r < -0.3
                      else "Moderate negative" if r < -0.1
                      else "Weak/No")
            print(f"  {feat:30s}  r={r:+.4f}  p={p:.2e}  {stars:4s}  → {interp} correlation")
    print("=" * 75)
    print("  Significance: *** p<0.001  ** p<0.01  * p<0.05  ns = not significant")


### Interpretation of Pearson Correlation Results

The Pearson correlation analysis reveals the linear associations between individual features and the binary sprint risk label.
Features such as `completion_rate` and `punt_rate` are expected to show the strongest correlations with the risk label,
given that they are directly used in its computation. However, `velocity_variance`, `schedule_variance`, and `story_point_delta`
provide independent signals of sprint instability that are valuable for predictive modelling.

It is important to note that Pearson correlation measures only **linear** relationships.
Non-linear interactions between features (captured by tree-based ensemble models) may yield
richer predictive signals than the correlation coefficients alone suggest.
High p-values for certain features (p > 0.05) indicate no statistically significant linear relationship,
though those features may still contribute to model performance through interactions with other variables.


## Cell 10 — Train / Validation / Test Split

Split `merged_train_data` into:
- **70% Training** — used to fit models
- **15% Validation** — used for hyperparameter selection and early stopping
- **15% Test** — held out for final unbiased evaluation

`StandardScaler` is fitted **only** on the training set and applied to all three splits.
Stratification on the target ensures class proportions are preserved across all splits.


In [ ]:
# ── Define features and target ───────────────────────────────────────────────
if len(merged_train_data) == 0:
    raise ValueError("merged_train_data is empty — re-run Cells 7 & 8.")

FEATURE_COLS = [c for c in merged_train_data.columns if c != TARGET_COL]
X = merged_train_data[FEATURE_COLS].values.astype(np.float64)
y = merged_train_data[TARGET_COL].values.astype(int)

print(f"Features matrix X: {X.shape}")
print(f"Target vector  y: {y.shape}")
print(f"Class distribution: {dict(zip(*np.unique(y, return_counts=True)))}")
print(f"\nFeature columns ({len(FEATURE_COLS)}):")
print(FEATURE_COLS)


In [ ]:
# ── 70/15/15 stratified split ─────────────────────────────────────────────────
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=42)

val_fraction = 0.15 / 0.85   # 15% of full = val / 85% of full
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=val_fraction, stratify=y_temp, random_state=42)

# ── Fit scaler on TRAIN only ──────────────────────────────────────────────────
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

# ── Shapes ────────────────────────────────────────────────────────────────────
print("Split shapes:")
print(f"  X_train : {X_train_s.shape}  |  y_train : {y_train.shape}")
print(f"  X_val   : {X_val_s.shape}  |  y_val   : {y_val.shape}")
print(f"  X_test  : {X_test_s.shape}  |  y_test  : {y_test.shape}")
print(f"  Total   : {X_train_s.shape[0]+X_val_s.shape[0]+X_test_s.shape[0]}")

# ── Class distribution per split ──────────────────────────────────────────────
print("\nClass distribution (label: count | %):")
for split_name, y_split in [("Train", y_train), ("Val", y_val), ("Test", y_test)]:
    unique, counts = np.unique(y_split, return_counts=True)
    dist_str = "  ".join([f"{u}: {c} ({c/len(y_split)*100:.1f}%)"
                           for u, c in zip(unique, counts)])
    print(f"  {split_name:6s}: {dist_str}")

# Save scaler
joblib.dump(scaler, CONTENT_DIR / "scaler.pkl")
print("\n✅ Scaler saved to /content/scaler.pkl")


## Cell 11 — Model Training

Train four classifiers on the scaled training set:
1. **Logistic Regression** — linear baseline with L2 regularisation
2. **Random Forest** — ensemble of decision trees with bagging
3. **Gradient Boosting** — sequential ensemble with boosting
4. **Support Vector Classifier (SVC)** — kernel-based classifier with RBF kernel

For each model: training accuracy, validation accuracy, and classification report are printed.


In [ ]:
import time

trained_models = {}

model_configs = [
    ("Logistic Regression",  LogisticRegression(max_iter=1000, random_state=42, C=1.0, solver="lbfgs")),
    ("Random Forest",        RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)),
    ("Gradient Boosting",    GradientBoostingClassifier(n_estimators=200, learning_rate=0.1,
                                                        max_depth=5, random_state=42)),
    ("SVC",                  SVC(kernel="rbf", C=1.0, gamma="scale",
                                 probability=True, random_state=42)),
]

for model_name, model in model_configs:
    print(f"\n{'='*60}")
    print(f"  Training: {model_name}")
    print(f"{'='*60}")
    t0 = time.time()

    model.fit(X_train_s, y_train)
    elapsed = time.time() - t0

    # Predictions
    y_pred_train = model.predict(X_train_s)
    y_pred_val   = model.predict(X_val_s)

    train_acc = accuracy_score(y_train, y_pred_train)
    val_acc   = accuracy_score(y_val,   y_pred_val)
    train_f1  = f1_score(y_train, y_pred_train, average="weighted")
    val_f1    = f1_score(y_val,   y_pred_val,   average="weighted")

    print(f"  Training time : {elapsed:.2f}s")
    print(f"  Train Accuracy: {train_acc:.4f}  |  Val Accuracy: {val_acc:.4f}")
    print(f"  Train F1      : {train_f1:.4f}  |  Val F1      : {val_f1:.4f}")
    print(f"\n  Validation Classification Report:")
    print(classification_report(y_val, y_pred_val, target_names=["No Risk", "At Risk"]))

    trained_models[model_name] = {
        "model": model,
        "train_acc": train_acc,
        "val_acc": val_acc,
        "train_f1": train_f1,
        "val_f1": val_f1,
        "elapsed": elapsed,
    }

print("\n✅ All models trained.")


## Cell 12 — Training vs Validation Loss Over Iterations

Track the learning curves for **Logistic Regression** and **Gradient Boosting**:
- **Logistic Regression**: re-trained incrementally from 10 to 500 iterations, recording log-loss at each checkpoint
- **Gradient Boosting**: uses `staged_predict` to record log-loss per estimator stage

These curves help diagnose overfitting (train loss decreasing while val loss increases) or
underfitting (both losses remaining high).


In [ ]:
# ── Logistic Regression — Log Loss per iteration checkpoint ─────────────────
print("Computing LR learning curve (log-loss checkpoints)...")
lr_train_losses = []
lr_val_losses   = []
lr_iters        = list(range(10, 510, 20))

for n_iter in lr_iters:
    lr_ckpt = LogisticRegression(max_iter=n_iter, random_state=42,
                                 C=1.0, solver="lbfgs", warm_start=False)
    lr_ckpt.fit(X_train_s, y_train)
    train_prob = lr_ckpt.predict_proba(X_train_s)
    val_prob   = lr_ckpt.predict_proba(X_val_s)
    # Clamp probabilities to avoid log(0)
    eps = 1e-7
    train_prob = np.clip(train_prob, eps, 1 - eps)
    val_prob   = np.clip(val_prob,   eps, 1 - eps)
    lr_train_losses.append(log_loss(y_train, train_prob))
    lr_val_losses.append(log_loss(y_val,   val_prob))

print(f"  LR: min train loss={min(lr_train_losses):.4f}, min val loss={min(lr_val_losses):.4f}")


In [ ]:
# ── Gradient Boosting — staged log-loss ──────────────────────────────────────
print("Computing GB staged loss (this may take ~30s)...")
gb_model_staged = GradientBoostingClassifier(
    n_estimators=200, learning_rate=0.1, max_depth=5, random_state=42)
gb_model_staged.fit(X_train_s, y_train)

gb_train_losses = []
gb_val_losses   = []
eps = 1e-7

for train_pred, val_pred in zip(
        gb_model_staged.staged_predict_proba(X_train_s),
        gb_model_staged.staged_predict_proba(X_val_s)):
    tp = np.clip(train_pred, eps, 1-eps)
    vp = np.clip(val_pred,   eps, 1-eps)
    gb_train_losses.append(log_loss(y_train, tp))
    gb_val_losses.append(log_loss(y_val,   vp))

gb_stages = list(range(1, len(gb_train_losses) + 1))
print(f"  GB: min train loss={min(gb_train_losses):.4f}, min val loss={min(gb_val_losses):.4f}")


In [ ]:
# ── Plot learning curves ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Learning Curves — Training vs Validation Log Loss",
             fontsize=14, fontweight="bold")

# LR subplot
axes[0].plot(lr_iters, lr_train_losses, color="#3498DB", linewidth=2,
             marker="o", markersize=4, label="Train Loss")
axes[0].plot(lr_iters, lr_val_losses,   color="#E74C3C", linewidth=2,
             linestyle="--", marker="s", markersize=4, label="Val Loss")
axes[0].set_title("Logistic Regression", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Iterations")
axes[0].set_ylabel("Log Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)
# Annotate convergence zone
best_lr_iter = lr_iters[np.argmin(lr_val_losses)]
axes[0].axvline(best_lr_iter, color="green", linestyle=":",
                label=f"Best Val @ iter {best_lr_iter}")
axes[0].legend()

# GB subplot
axes[1].plot(gb_stages, gb_train_losses, color="#3498DB", linewidth=2,
             label="Train Loss")
axes[1].plot(gb_stages, gb_val_losses,   color="#E74C3C", linewidth=2,
             linestyle="--", label="Val Loss")
axes[1].set_title("Gradient Boosting", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Number of Estimators")
axes[1].set_ylabel("Log Loss")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
best_gb_stage = gb_stages[np.argmin(gb_val_losses)]
axes[1].axvline(best_gb_stage, color="green", linestyle=":",
                label=f"Best Val @ stage {best_gb_stage}")
axes[1].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "learning_curves.png", bbox_inches="tight")
plt.show()

# ── Commentary ────────────────────────────────────────────────────────────────
print("\n📝 Commentary:")
lr_overfit  = lr_val_losses[-1] > min(lr_val_losses) * 1.05
gb_overfit  = gb_val_losses[-1] > min(gb_val_losses) * 1.05
lr_underfit = min(lr_train_losses) > 0.4
gb_underfit = min(gb_train_losses) > 0.4
print(f"  Logistic Regression — {'Possible overfitting detected' if lr_overfit else 'No overfitting'}",
      f"| {'Possible underfitting' if lr_underfit else 'Adequate fit'}")
print(f"  Gradient Boosting   — {'Possible overfitting detected' if gb_overfit else 'No overfitting'}",
      f"| {'Possible underfitting' if gb_underfit else 'Adequate fit'}")


## Cell 13 — Model Evaluation

Comprehensive evaluation of all four trained models on the **held-out test set** (15% of data):
- Accuracy, Precision (weighted), Recall (weighted), F1 (weighted), ROC-AUC, CV-F1 (5-fold)
- Summary comparison table
- Confusion matrix grid (2×2 heatmaps)
- ROC curves (all models on one chart)
- Random Forest feature importance (top 20)
- Best model saved as `best_model.pkl`


In [ ]:
# ── Compute all metrics on test set ──────────────────────────────────────────
from sklearn.preprocessing import label_binarize

eval_results = []
n_classes = len(np.unique(y))

cv_skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for model_name, info in trained_models.items():
    model = info["model"]
    y_pred = model.predict(X_test_s)

    # Probabilities
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test_s)
    else:
        # Decision function for SVM
        y_df = model.decision_function(X_test_s)
        y_prob = np.column_stack([-y_df, y_df]) if y_df.ndim == 1 else y_df

    # Metrics
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average="weighted", zero_division=0)
    rec  = recall_score(y_test,   y_pred, average="weighted",  zero_division=0)
    f1   = f1_score(y_test,       y_pred, average="weighted",  zero_division=0)

    # ROC-AUC
    try:
        if n_classes == 2:
            roc_auc = roc_auc_score(y_test, y_prob[:, 1])
        else:
            roc_auc = roc_auc_score(y_test, y_prob, multi_class="ovr", average="weighted")
    except Exception:
        roc_auc = float("nan")

    # 5-fold CV F1 on combined train+val set
    X_trainval = np.vstack([X_train_s, X_val_s])
    y_trainval = np.concatenate([y_train, y_val])
    try:
        cv_f1_scores = cross_val_score(model, X_trainval, y_trainval,
                                       cv=cv_skf, scoring="f1_weighted", n_jobs=-1)
        cv_f1 = cv_f1_scores.mean()
    except Exception:
        cv_f1 = float("nan")

    eval_results.append({
        "Model": model_name,
        "Accuracy": round(acc,  4),
        "Precision": round(prec, 4),
        "Recall":    round(rec,  4),
        "F1":        round(f1,   4),
        "ROC-AUC":   round(roc_auc, 4) if not np.isnan(roc_auc) else "N/A",
        "CV-F1 (5)": round(cv_f1, 4)   if not np.isnan(cv_f1)   else "N/A",
    })
    trained_models[model_name]["y_pred"]  = y_pred
    trained_models[model_name]["y_prob"]  = y_prob
    trained_models[model_name]["test_f1"] = f1

eval_df = pd.DataFrame(eval_results)
print("\n" + "="*75)
print("  MODEL EVALUATION SUMMARY — TEST SET")
print("="*75)
print(eval_df.to_string(index=False))
print("="*75)


In [ ]:
# ── Confusion Matrices — 2×2 Grid ────────────────────────────────────────────
model_names_list = list(trained_models.keys())
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle("Confusion Matrices — Test Set", fontsize=14, fontweight="bold")

for ax, model_name in zip(axes.flat, model_names_list):
    y_pred = trained_models[model_name]["y_pred"]
    cm_arr = confusion_matrix(y_test, y_pred)
    labels = ["No Risk", "At Risk"] if n_classes == 2 else [str(i) for i in range(n_classes)]
    sns.heatmap(cm_arr, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=labels, yticklabels=labels,
                linewidths=1, linecolor="white", annot_kws={"size": 12})
    ax.set_title(model_name, fontsize=11, fontweight="bold")
    ax.set_xlabel("Predicted Label")
    ax.set_ylabel("True Label")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrices.png", bbox_inches="tight")
plt.show()


In [ ]:
# ── ROC Curves — All Models on One Chart ────────────────────────────────────
if n_classes == 2:
    fig, ax = plt.subplots(figsize=(10, 7))
    roc_colors = ["#3498DB", "#2ECC71", "#E74C3C", "#9B59B6"]
    for color, model_name in zip(roc_colors, model_names_list):
        y_prob = trained_models[model_name]["y_prob"]
        try:
            fpr, tpr, _ = roc_curve(y_test, y_prob[:, 1])
            roc_auc_val = auc(fpr, tpr)
            ax.plot(fpr, tpr, color=color, linewidth=2,
                    label=f"{model_name} (AUC={roc_auc_val:.3f})")
        except Exception:
            pass

    ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="Random Classifier")
    ax.fill_between([0, 1], [0, 1], alpha=0.05, color="grey")
    ax.set_title("ROC Curves — All Models (Test Set)", fontsize=12, fontweight="bold")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "roc_curves.png", bbox_inches="tight")
    plt.show()
else:
    print("ROC curve plot skipped (multiclass — use OvR AUC in eval table).")


In [ ]:
# ── Feature Importance — Random Forest ───────────────────────────────────────
rf_model = trained_models.get("Random Forest", {}).get("model")
if rf_model is not None and hasattr(rf_model, "feature_importances_"):
    importances = rf_model.feature_importances_
    feat_imp_df = pd.DataFrame({
        "Feature": FEATURE_COLS,
        "Importance": importances
    }).sort_values("Importance", ascending=False).head(20)

    fig, ax = plt.subplots(figsize=(10, 8))
    colors_fi = plt.cm.RdYlGn(np.linspace(0.2, 0.85, len(feat_imp_df)))[::-1]
    ax.barh(feat_imp_df["Feature"][::-1], feat_imp_df["Importance"][::-1],
            color=colors_fi, edgecolor="white")
    ax.set_title("Random Forest — Top 20 Feature Importances",
                 fontsize=12, fontweight="bold")
    ax.set_xlabel("Gini Importance")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "feature_importance_rf.png", bbox_inches="tight")
    plt.show()

    print("\nTop 10 features by Random Forest importance:")
    print(feat_imp_df.head(10).to_string(index=False))


In [ ]:
# ── Save best model and scaler ────────────────────────────────────────────────
best_model_name = max(trained_models, key=lambda k: trained_models[k].get("test_f1", 0))
best_model_obj  = trained_models[best_model_name]["model"]

best_model_path = CONTENT_DIR / "best_model.pkl"
scaler_path     = CONTENT_DIR / "scaler.pkl"

joblib.dump(best_model_obj, best_model_path)
joblib.dump(scaler, scaler_path)

# Copy to output dir too
joblib.dump(best_model_obj, OUTPUT_DIR / "best_model.pkl")
joblib.dump(scaler, OUTPUT_DIR / "scaler.pkl")

print(f"\n🏆 Best Model: {best_model_name}")
print(f"   Test F1: {trained_models[best_model_name]['test_f1']:.4f}")
print(f"   Saved to: {best_model_path}")
print(f"\n✅ Evaluation complete. All plots saved to {OUTPUT_DIR}")


## Cell 14 — Live Demo: AI-Enhanced Predictive Risk Management

**Four realistic sprint scenarios** are fed through the trained best model to demonstrate
the end-to-end prediction pipeline. Each scenario represents a distinct risk profile,
from a healthy sprint to a crisis state.

The `predict_sprint_risk()` function:
1. Loads the saved `best_model.pkl` and `scaler.pkl`
2. Aligns the input feature dictionary to the trained schema
3. Returns risk label, class index, probability, and an actionable recommendation


In [ ]:
# ── Prediction function ───────────────────────────────────────────────────────

RISK_LABELS = {0: "Low", 1: "At Risk"}
RISK_RECOMMENDATIONS = {
    "Low":     "✅ Sprint is on track. Monitor velocity weekly.",
    "Medium":  "⚠️  Caution: review backlog and re-estimate story points.",
    "High":    "🚨 Intervention needed: escalate to project manager and re-plan sprint.",
    "Critical":"🔴 Sprint in crisis: halt new work, conduct emergency retrospective.",
    "At Risk": "⚠️  Sprint showing risk signals: review completion rate, punt rate, and schedule."
}

def predict_sprint_risk(sprint_input_dict,
                        model_path=None,
                        scaler_path=None,
                        feature_cols=None):
    """
    Predict sprint risk from a feature dictionary.
    Returns dict: risk_label, risk_class, probability, recommendation.
    """
    # Load model and scaler
    _model_path  = model_path  or str(CONTENT_DIR / "best_model.pkl")
    _scaler_path = scaler_path or str(CONTENT_DIR / "scaler.pkl")

    try:
        _model  = joblib.load(_model_path)
        _scaler = joblib.load(_scaler_path)
    except Exception as e:
        # Fall back to in-memory objects
        _model  = best_model_obj
        _scaler = scaler

    _feature_cols = feature_cols or FEATURE_COLS

    # Build feature vector aligned to training schema
    feature_vector = np.zeros(len(_feature_cols))
    for i, col in enumerate(_feature_cols):
        if col in sprint_input_dict:
            feature_vector[i] = float(sprint_input_dict[col])

    feature_vector = feature_vector.reshape(1, -1)
    feature_scaled = _scaler.transform(feature_vector)

    # Predict
    risk_class = int(_model.predict(feature_scaled)[0])

    if hasattr(_model, "predict_proba"):
        proba = _model.predict_proba(feature_scaled)[0]
        probability = float(proba[risk_class]) if risk_class < len(proba) else float(proba[-1])
    elif hasattr(_model, "decision_function"):
        df_val = _model.decision_function(feature_scaled)[0]
        probability = float(1 / (1 + np.exp(-abs(df_val))))
    else:
        probability = 1.0

    risk_label = RISK_LABELS.get(risk_class, "High" if risk_class >= 2 else "At Risk")
    recommendation = RISK_RECOMMENDATIONS.get(risk_label, RISK_RECOMMENDATIONS["At Risk"])

    return {
        "risk_label":     risk_label,
        "risk_class":     risk_class,
        "probability":    probability,
        "recommendation": recommendation,
    }

print("✅ predict_sprint_risk() function ready.")


In [ ]:
# ── Define the four demo scenarios ───────────────────────────────────────────
scenarios = [
    {
        "title": "Scenario 1 — Low Risk Sprint (healthy)",
        "color": "\033[92m",  # green
        "input": {
            "completion_rate":  0.92,
            "velocity":         45,
            "backlog_growth":   3,
            "punt_rate":        0.05,
            "schedule_variance": -1,
            "avg_story_points": 5.2,
            "story_point_delta": 0.1,
            "defect_density":   0.02,
        }
    },
    {
        "title": "Scenario 2 — Medium Risk Sprint (signs of drift)",
        "color": "\033[93m",  # yellow
        "input": {
            "completion_rate":  0.68,
            "velocity":         28,
            "backlog_growth":   12,
            "punt_rate":        0.22,
            "schedule_variance": 3,
            "avg_story_points": 8.1,
            "story_point_delta": 1.4,
            "defect_density":   0.07,
        }
    },
    {
        "title": "Scenario 3 — High Risk Sprint (clear warning)",
        "color": "\033[91m",  # red
        "input": {
            "completion_rate":  0.45,
            "velocity":         15,
            "backlog_growth":   28,
            "punt_rate":        0.41,
            "schedule_variance": 9,
            "avg_story_points": 11.3,
            "story_point_delta": 3.8,
            "defect_density":   0.18,
        }
    },
    {
        "title": "Scenario 4 — Critical Risk Sprint (crisis)",
        "color": "\033[91m",  # bold red
        "input": {
            "completion_rate":  0.22,
            "velocity":         6,
            "backlog_growth":   47,
            "punt_rate":        0.72,
            "schedule_variance": 21,
            "avg_story_points": 18.9,
            "story_point_delta": 7.2,
            "defect_density":   0.41,
        }
    },
]


In [ ]:
# ── Run predictions and print formatted report ───────────────────────────────
RESET = "\033[0m"
BOLD  = "\033[1m"

print(f"\n{'='*70}")
print(f"  🤖 AgileRisk AI — LIVE PREDICTION DEMO")
print(f"  Model: {best_model_name}")
print(f"{'='*70}")

results = []
for scenario in scenarios:
    result = predict_sprint_risk(scenario["input"])
    results.append(result)
    c = scenario["color"]

    print(f"\n{BOLD}{'─'*65}{RESET}")
    print(f"{BOLD}{c}  {scenario['title']}{RESET}")
    print(f"{'─'*65}")

    # Input metrics
    print(f"  INPUT METRICS:")
    for k, v in scenario["input"].items():
        print(f"    {k:25s}: {v}")

    # Prediction output
    prob_pct = result['probability'] * 100
    bar_filled = int(prob_pct / 5)  # out of 20 blocks
    bar_empty  = 20 - bar_filled
    bar_str    = f"[{'█' * bar_filled}{'░' * bar_empty}] {prob_pct:.1f}%"

    print(f"\n  PREDICTION:")
    print(f"    Risk Label  : {c}{BOLD}{result['risk_label']}{RESET}")
    print(f"    Risk Class  : {result['risk_class']}")
    print(f"    Probability : {bar_str}")
    print(f"    Recommendation: {result['recommendation']}")

print(f"\n{'='*70}")


In [ ]:
# ── Gauge visualisation — 2×2 grid of semicircular gauges ─────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 12),
                          subplot_kw={"projection": "polar"})
fig.suptitle("AgileRisk AI — Sprint Risk Probability Gauges",
             fontsize=14, fontweight="bold", y=1.01)

gauge_colors_map = {
    "Low":     ("#2ECC71", "#27AE60"),
    "At Risk": ("#F0A500", "#E67E22"),
    "High":    ("#E67E22", "#E74C3C"),
    "Critical":("#E74C3C", "#C0392B"),
}

for ax, scenario, result in zip(axes.flat, scenarios, results):
    prob = result["probability"]
    label = result["risk_label"]
    col_outer, col_inner = gauge_colors_map.get(label, ("#95A5A6", "#7F8C8D"))

    # Semicircular gauge: theta from pi to 0 (left to right)
    theta_max = np.pi - (prob * np.pi)
    theta_fill = np.linspace(np.pi, theta_max, 200)
    theta_bg   = np.linspace(np.pi, 0,         200)

    # Background arc (grey)
    ax.plot(theta_bg, np.ones_like(theta_bg) * 0.8, color="#ECEFF1",
            linewidth=20, solid_capstyle="round")
    # Fill arc (risk colour)
    ax.plot(theta_fill, np.ones_like(theta_fill) * 0.8, color=col_outer,
            linewidth=20, solid_capstyle="round")
    # Needle
    needle_theta = theta_max
    ax.annotate("", xy=(needle_theta, 0.75), xytext=(0, 0),
                arrowprops=dict(arrowstyle="->", color=col_inner,
                                lw=3, mutation_scale=18))
    # Centre text
    ax.text(0, 0, f"{prob*100:.1f}%",
            ha="center", va="center", fontsize=16, fontweight="bold", color=col_inner)
    ax.text(np.pi / 2, 1.15, label,
            ha="center", va="bottom", fontsize=12, fontweight="bold", color=col_outer)

    ax.set_ylim(0, 1.3)
    ax.set_xlim(0, np.pi)
    ax.set_theta_zero_location("S")
    ax.set_theta_direction(-1)
    ax.set_rticks([])
    ax.set_xticks([0, np.pi / 4, np.pi / 2, 3 * np.pi / 4, np.pi])
    ax.set_xticklabels(["High\nRisk", "", "Med", "", "Low\nRisk"], fontsize=7)
    ax.spines["polar"].set_visible(False)
    ax.set_title(scenario["title"].split("—")[0].strip(),
                 fontsize=10, pad=15, fontweight="bold")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "risk_gauges.png", bbox_inches="tight")
plt.show()
print("\n✅ Live demo complete. Gauge chart saved.")


## Cell 15 — Conclusions

### Key Findings from EDA

The exploratory analysis across the five NASA PROMISE datasets revealed consistent class imbalance —
typically fewer than 25% of modules are defective — which necessitated stratified splitting.
The cm1 Pearson correlation heatmap showed that `branchCount`, `v(g)`, and `ev(g)` (cyclomatic complexity metrics)
are among the strongest predictors of defect presence, consistent with the software engineering literature.

Sprint velocity analysis across the four Agile projects (Mesos, Spring XD, Aurora, Usergrid)
revealed high variance in completion rates and a tendency for backlog growth to correlate with sprint risk.
The Aurora project exhibited the highest punt rates, suggesting frequent re-planning.
The engineered `sprint_risk_label` (completion_rate < 0.6 OR punt_rate > 0.3 OR schedule_variance > 2)
successfully identifies sprints warranting intervention across all four projects.

---

### Best Performing Model

Ensemble models (Random Forest and Gradient Boosting) consistently outperformed the linear baseline
(Logistic Regression) and kernel-based SVC on this dataset, achieving higher F1 scores on the test set.
The best model (see Cell 13 for exact figures) was saved as `best_model.pkl` and is used throughout the live demo.
Random Forest feature importances confirm that `completion_rate`, `punt_rate`, and `schedule_variance` —
the engineered sprint risk proxy features — are the most influential predictors, validating the feature engineering choices.

---

### What the Live Demo Demonstrates

The four demo scenarios in Cell 14 demonstrate that the AI pipeline can successfully discriminate between
risk levels in real-time, providing not only a class prediction but also a calibrated probability and
an actionable recommendation. This validates the core dissertation objective: that AI can enhance
predictive risk management in Agile IT projects by providing early warning signals before a sprint fails.

---

### Limitations of the Study

1. **Dataset temporal mismatch**: NASA PROMISE defect datasets are from space systems (1990s–2000s) and may not
   generalise directly to modern Agile web projects.
2. **Sprint risk label engineering**: The binary label is rule-based, introducing human bias into the target variable.
   Future work should explore expert-labelled risk annotations.
3. **Merged dataset heterogeneity**: Concatenating NASA (module-level), sprint-level, and project-level datasets
   required zero-padding of irrelevant features, which may reduce model interpretability.
4. **Class imbalance**: The NASA datasets exhibit significant class imbalance that SMOTE or cost-sensitive learning could address.
5. **No temporal cross-validation**: Sprint datasets are time-ordered; random splitting may inflate performance estimates.

---

### Future Work Recommendations

- Apply **SMOTE** (Synthetic Minority Oversampling Technique) to address class imbalance
- Implement **LSTM / temporal models** to exploit the sequential nature of sprint data
- Integrate **real-time Jira/Agile tool APIs** for live sprint monitoring
- Conduct **hyperparameter tuning** using Bayesian optimisation (Optuna / Hyperopt)
- Extend to **multiclass risk** (Low / Medium / High / Critical) aligned with the project_risk_raw_dataset
- Deploy the model as a **REST API** (FastAPI) integrated into existing Agile project management tools

---

### Dissertation Objectives Met

| Objective | Status |
|---|---|
| Identify key AI techniques applicable to Agile risk management | ✅ Achieved |
| Develop a feature engineering pipeline for sprint risk proxies | ✅ Achieved |
| Train and evaluate multiple ML classifiers | ✅ Achieved |
| Demonstrate real-time risk prediction on new sprint data | ✅ Achieved |
| Produce a deployable artefact (App.py + best_model.pkl) | ✅ Achieved |

---

*Manfred Oppong | B01814357 | MSc IT with Project Management | University of the West of Scotland | 2026*
